### RAG Pipelines = Data Ingestion to Vector DB Pipeline

In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [16]:
### Read all the pdf's inside the drectory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for file in pdf_files:
        print(f"\nProcessing: {file.name}")
        try:
            loader = PyPDFLoader(str(file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages") 

        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

## Process all PDFs in the data directory
all_pdfs_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: embeddings.pdf
Loaded 4 pages

Processing: attention.pdf
Loaded 4 pages

Processing: object_detection.pdf
Loaded 4 pages

Processing: proposal.pdf
Loaded 3 pages

Total documents loaded: 15


In [20]:
all_pdfs_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-11-17T20:22:05-03:00', 'author': 'Mariana Costa, Thiago Ribeiro', 'title': 'Vector Embeddings for Semantic Search: Concepts, Models and Evaluation', 'subject': 'Information Retrieval', 'keywords': 'embeddings, semantic search, vector database, cosine similarity, RAG', 'moddate': '2026-11-17T20:22:05-03:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '../data/pdf/embeddings.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'embeddings.pdf', 'file_type': 'pdf'}, page_content='Vector Embeddings for Semantic Search:\nConcepts, Models and Evaluation\nMariana Costa Thiago Ribeiro\nDepartment of Computer Science, Federal Technology University\nJune 2024\nAbstract\nEmbeddings are dense numerical vectors that represent the meaning of words, sentences,\nimages, or docume

In [ ]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f'Split {len(documents)} documents into {len(split_docs)} chunks')

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f'Content: {split_docs[0].page_content[:200]}...')
        print(f'Metadata: {split_docs[0].metadata}')

    return split_docs


In [ ]:
chunks = split_documents(all_pdfs_documents)
chunks

Split 15 documents into 39 chunks

Example chunk:
Content: Vector Embeddings for Semantic Search:
Concepts, Models and Evaluation
Mariana Costa Thiago Ribeiro
Department of Computer Science, Federal Technology University
June 2024
Abstract
Embeddings are dens...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-11-17T20:22:05-03:00', 'author': 'Mariana Costa, Thiago Ribeiro', 'title': 'Vector Embeddings for Semantic Search: Concepts, Models and Evaluation', 'subject': 'Information Retrieval', 'keywords': 'embeddings, semantic search, vector database, cosine similarity, RAG', 'moddate': '2026-11-17T20:22:05-03:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '../data/pdf/embeddings.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'embeddings.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-11-17T20:22:05-03:00', 'author': 'Mariana Costa, Thiago Ribeiro', 'title': 'Vector Embeddings for Semantic Search: Concepts, Models and Evaluation', 'subject': 'Information Retrieval', 'keywords': 'embeddings, semantic search, vector database, cosine similarity, RAG', 'moddate': '2026-11-17T20:22:05-03:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '../data/pdf/embeddings.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'embeddings.pdf', 'file_type': 'pdf'}, page_content='Vector Embeddings for Semantic Search:\nConcepts, Models and Evaluation\nMariana Costa Thiago Ribeiro\nDepartment of Computer Science, Federal Technology University\nJune 2024\nAbstract\nEmbeddings are dense numerical vectors that represent the meaning of words, sentences,\nimages, or docume